1 CÀI THƯ VIỆN
--------------

In [1]:
pip install transformers datasets accelerate seqeval evaluate underthesea

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 97.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 49.4 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=d4fe9ea54576b6efa147c2ceda45044f80c1130faab909c40883fddc424fe1c1
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
Note: you may need to restart the kernel to use updated packages.


2 LOAD THƯ VIỆN
---------------

In [39]:
import re
import torch
import unicodedata
import evaluate
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import Trainer
from transformers import AutoModelForTokenClassification
from transformers import TrainingArguments
from transformers import DataCollatorForTokenClassification
from sklearn.metrics import (precision_recall_fscore_support,accuracy_score)

3 LOAD DATA
-----------

In [3]:
df = pd.read_csv("/kaggle/input/datasets/giba1010/multilexnorm-2026/train_vi.csv")
df.head() # in ra 5 sample đầu

,raw,norm,lang,split
0,['thích' 'anh' 'cá' 'mập' 'k'],['thích' 'anh' 'cá' 'mập' 'không'],vi,train
1,['cứ' 'ngây' 'thơ' 'thế' 'thoai' ':))'],['cứ' 'ngây' 'thơ' 'thế' 'thôi' ':))'],vi,train
2,['bà' 'Nghê' 'xinh' 'vậy' 'mà' 't' 'thấy' 'k' ...,['bà' 'Nghê' 'xinh' 'vậy' 'mà' 'tôi' 'thấy' 'k...,vi,train
3,['Ê' 'k' 'khóc' 'được' 'làm' 'thế' 'nào' 'má' ...,['Ê' 'không' 'khóc' 'được' 'làm' 'thế' 'nào' '...,vi,train
4,['Có' 'biến' 'gì' 'hong' 'dẫy' ':))'],['Có' 'biến' 'gì' 'không' 'vậy' ':))'],vi,train


4 PARSE TOKEN LIST
-----------------
Model PhoBERT cần 1 list token thật nhưng hiện tại thì dataset chỉ là dạng một cái string giống như list nên chúng ta phải chuyển về list token thật.

In [4]:
def normalize_unicode(text):
    return unicodedata.normalize("NFC", text)

def parse_tokens(text):
    text = normalize_unicode(text)
    return re.findall(r"'(.*?)'", text)

In [5]:
parse_tokens("['thích' 'anh' 'cá' 'mập' 'k']") # test thử xem đã convert sang python token chưa

['thích', 'anh', 'cá', 'mập', 'k']

Kết quả: Đã chuyển dạng dạng token của python rồi.

5 CONVERT DATASET
-----------------

In [6]:
df["raw_tokens"] = df["raw"].apply(parse_tokens) # Áp dụng cho cả cột raw của dataset
df["norm_tokens"] = df["norm"].apply(parse_tokens) # Áp dụng cho cả cột norm của dataset

6 TẠO LABELS
------------
Ý tưởng là: nếu raw = norm thì 0 còn nếu raw khác norm thì 1

Bước này để làm gì và tại sao lại có?
- Bước này là đánh số để biến bài toán này thành bài toán supervised learning. Có nghĩ là Model này phải biết token nào đúng và token nào là sai.
- Vói ý nghĩa là label 1 là đúng, toket được giữ nguyên. Nhưng nếu 0 là sai thì token cần được normalize.

In [7]:
def create_labels(raw_tokens, norm_tokens):
    labels = []

    for r, n in zip(raw_tokens, norm_tokens):
        if r == n:
            labels.append(0)  # OK
        else:
            labels.append(1)  # ERR

    return labels

In [8]:
df["labels"] = df.apply(
    lambda x: create_labels(
        x["raw_tokens"],
        x["norm_tokens"]
    ),
    axis=1
)

7 TẠO DICTIONARY NORMALIZE
--------------------------

Bước này là tạo dictionary cho dataset (slang)

In [9]:
NORMALIZE_DICT = {}

# slang/noisy token -> token chuẩn

# Duyệt từng câu trong dataset
for raw_tokens, norm_tokens in zip(
    df["raw_tokens"], # câu raw
    df["norm_tokens"] # câu normalized
):
     # Duyệt từng token trong câu
    for r, n in zip(raw_tokens, norm_tokens):
        # Nếu token khác nhau
        # nghĩa là token raw cần normalize
        if r != n:
            # lưu mapping
            # ví dụ:
            # "ko" -> "không"
            NORMALIZE_DICT[r] = n

8 PHOBERT TOKENIZER
-------------------

Gọi thư viện và chuyển text -> token ids vì model không đọc được chữ đâu. Chỉ đọc được số thôi.

In [10]:
tokenizer = AutoTokenizer.from_pretrained(
    "vinai/phobert-base"
)

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

9 CHUẨN BỊ INPUT TOKENIZER + ALIGN LABELS
-----------------------------------------

Dataset gốc chưa phải là format mà dataset có thể train dược. Phải biến dataset thành: input cho model + labels
- Input là raw
- labels là đáp án cho từng token

Tại sao?
- Vì học có giám sát cần input và trong input đó thì token nào sai.

In [11]:
def tokenize_and_align_labels(example):
     # example["raw_tokens"]
    # ví dụ:
    # ['t', 'ko', 'bt']

    tokenized_inputs = tokenizer(

        # list token đã tách sẵn
        example["raw_tokens"],

        # báo cho tokenizer biết:
        # đây là tokenized words rồi
        is_split_into_words=True,

        # cắt nếu quá dài
        truncation=True,

        # độ dài tối đa
        max_length=128
    )
    # labels gốc
    labels = example["labels"]

    # thêm ignore cho special tokens
    aligned_labels = [-100] + labels + [-100]

    # cắt nếu lệch chiều dài
    aligned_labels = aligned_labels[:len(tokenized_inputs["input_ids"])]

    tokenized_inputs["labels"] = aligned_labels

    return tokenized_inputs

10 CONVERT SANG HUGGINGFACE DATASET
----------------------------------

- Đang dùng trainer của Huggingface nên phải convert dataset sang huggingface luôn. 
- Bên cạnh đó huggingface cũng hỗ trợ các đoạn code batching, padding, format training loop.
- Nếu không dùng huggingface thì ta sẽ phải tự tại các đoạn code bên trên bằng Pytorch thuần.

In [13]:
dataset = Dataset.from_pandas(df)

In [14]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels
)

Map:   0%|          | 0/8371 [00:00<?, ? examples/s]

11 TẠO DATA COLLATOR
--------------------

In [15]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

11 LOAD MODEL
-------------

In [16]:
model = AutoModelForTokenClassification.from_pretrained(
    "vinai/phobert-base",
    num_labels=2
)

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.bias                 | MISSING    | 
classifier.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


12 TRAINING ARGUMENTS
---------------------

Cấu hình cho model sẽ train như thế nào.

In [17]:
training_args = TrainingArguments(
    output_dir="./phoBERT_multilexnorm", # nơi lưu model. Sau train sẽ có
    eval_strategy="epoch", # evaluate mỗi epoch (sau khi train xong 1 epoch evaluate 1 lần)
    learning_rate=2e-5, # learning rate
    per_device_train_batch_size=8, # mỗi lần train thì model nhìn 8 samples
    per_device_eval_batch_size=8, # batch size eval
    num_train_epochs=5, # số epoch
    weight_decay=0.01, # phạt model nếu weights quá lớn
    save_strategy="epoch", # lưu model mỗi epoch
    load_best_model_at_end=True, # Load best model cuối training
    logging_dir="./logs", # nơi lưu training logs
    logging_steps=100 # log mỗi 100 steps
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


13 METRICS
----------

Đây là phần để đo xem model học tốt tới đâu.
- Tại sao có training loss rổi mà vẫn cần chỉ số Metrics này?

Metrics là gì? là các chỉ số đánh giá model như Accuracy, precision, recall, F1-score.

In [18]:
metric = evaluate.load("seqeval")

Compute metrics

In [19]:
def compute_metrics(p):

    predictions, labels = p

    # lấy label predict
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    # REMOVE -100


    for pred, lab in zip(predictions, labels):

        for p_, l_ in zip(pred, lab):

            # bỏ special tokens
            if l_ != -100:

                true_predictions.append(p_)
                true_labels.append(l_)

    # METRICS

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        true_predictions,

        # QUAN TRỌNG
        average="weighted",

        zero_division=0
    )

    accuracy = accuracy_score(
        true_labels,
        true_predictions
    )

    return {
        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

14 TRAINER
----------

Đây là nơi quả lý toàn bộ quá trình train model (nơi huấn luyện model của HuggingFace)

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    data_collator=data_collator
)

15 TRAIN
--------

In [21]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.382565,0.305108,0.938141,0.938207,0.938275,0.938141
2,0.272008,0.201912,0.962191,0.961954,0.961797,0.962191
3,0.224963,0.163879,0.970899,0.970520,0.970494,0.970899
4,0.203390,0.144228,0.973877,0.973884,0.973892,0.973877
5,0.185259,0.126364,0.978172,0.978065,0.978011,0.978172


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=2620, training_loss=0.2757111404688304, metrics={'train_runtime': 676.3146, 'train_samples_per_second': 61.887, 'train_steps_per_second': 3.874, 'total_flos': 883458624190800.0, 'train_loss': 0.2757111404688304, 'epoch': 5.0})

Nhận xét:
- Các chỉ số tổng thể rất đẹp và cho thấy model đang học ổn định.
- validation loss giảm liên tục từ 0.128 -> 0.054
- F1 tăng đều 97.5% -> 99%
- Precision ~ Recall ~ F1-score

ĐÃ XONG QUA TRÌNH TRAIN MODEL TRÊN TỆP TRAIN. TIẾP THEO SẼ PREDICT TRÊN TỆP VALIDATION
--------------------------------------------------------------------------------------

16 LOAD MODEL ĐÃ TRAIN
----------------------

In [66]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification
)

checkpoint_path = (
    "/kaggle/working/phoBERT_multilexnorm/checkpoint-2620"
)

tokenizer = AutoTokenizer.from_pretrained(
    "vinai/phobert-base",
    use_fast=False
)

model = AutoModelForTokenClassification.from_pretrained(
    checkpoint_path
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

17 LOAD VALIDATION DATA
----------------------

In [67]:
df_val = pd.read_csv("/kaggle/input/datasets/giba1010/multilexnorm-2026/validation_vi.csv")
df_val.head()

,raw,norm,lang,split
0,"['Bên' 'ni' 'đèo' 'thì' '""bả' 'sá"",' 'bên' 'tê...","['Bên' 'này' 'đèo' 'thì' '""bả' 'sá"",' 'bên' 'k...",vi,validation
1,['2' 'mày' 'bán' 'tao' 'ngồi' 'đếm' 'xiền' 'xe...,['2' 'mày' 'bán' 'tao' 'ngồi' 'đếm' 'tiền' 'xe...,vi,validation
2,['Vứt' 'con' 'đấy' 'đi' 'hành' 'hung' 'ng' 'kh...,['Vứt' 'con' 'đấy' 'đi' 'hành' 'hung' 'người' ...,vi,validation
3,['V' 'là' 'ai' 'ở' 'nhà' 'ôm' 'con' 'cũng' 'rả...,['Vậy' 'là' 'ai' 'ở' 'nhà' 'ôm' 'con' 'cũng' '...,vi,validation
4,['3' 'đứa' 'm' 'có' 'v' 'k'],['3' 'đứa' 'm' 'có' 'vậy' 'không'],vi,validation


18 PREPROCESS
--------------

In [69]:
df_val["raw_tokens"] = df_val["raw"].apply(parse_tokens) # Áp dụng cho cả cột raw của tệp validation
df_val["norm_tokens"] = df_val["norm"].apply(parse_tokens) # Áp dụng cho cả cột norm của validation

In [33]:
df_val.head()

,raw,norm,lang,split,raw_tokens,norm_tokens
0,"['Bên' 'ni' 'đèo' 'thì' '""bả' 'sá"",' 'bên' 'tê...","['Bên' 'này' 'đèo' 'thì' '""bả' 'sá"",' 'bên' 'k...",vi,validation,"[Bên, ni, đèo, thì, ""bả, sá"",, bên, tê, đèo, t...","[Bên, này, đèo, thì, ""bả, sá"",, bên, kia, đèo,..."
1,['2' 'mày' 'bán' 'tao' 'ngồi' 'đếm' 'xiền' 'xe...,['2' 'mày' 'bán' 'tao' 'ngồi' 'đếm' 'tiền' 'xe...,vi,validation,"[2, mày, bán, tao, ngồi, đếm, xiền, xem, có, đ...","[2, mày, bán, tao, ngồi, đếm, tiền, xem, có, đ..."
2,['Vứt' 'con' 'đấy' 'đi' 'hành' 'hung' 'ng' 'kh...,['Vứt' 'con' 'đấy' 'đi' 'hành' 'hung' 'người' ...,vi,validation,"[Vứt, con, đấy, đi, hành, hung, ng, khác, tron...","[Vứt, con, đấy, đi, hành, hung, người, khác, t..."
3,['V' 'là' 'ai' 'ở' 'nhà' 'ôm' 'con' 'cũng' 'rả...,['Vậy' 'là' 'ai' 'ở' 'nhà' 'ôm' 'con' 'cũng' '...,vi,validation,"[V, là, ai, ở, nhà, ôm, con, cũng, rảnh, háng,...","[Vậy, là, ai, ở, nhà, ôm, con, cũng, rảnh, hán..."
4,['3' 'đứa' 'm' 'có' 'v' 'k'],['3' 'đứa' 'm' 'có' 'vậy' 'không'],vi,validation,"[3, đứa, m, có, v, k]","[3, đứa, m, có, vậy, không]"


19 TẠO LABELS CHO VALIDATION
-----------

Giống với tạo labels cho tệp train.

In [70]:
df_val["labels"] = df_val.apply(
    lambda x: create_labels(
        x["raw_tokens"],
        x["norm_tokens"]
    ),
    axis=1
)

20 CONVERT SANG DATASET
-----------------------

convert dataset sang HugggingFace giống như tệp train

In [73]:
val_dataset = Dataset.from_pandas(df_val)

21 TOKENIZE VALIDATION
----------------------

In [74]:
tokenized_val_dataset = val_dataset.map(
    tokenize_and_align_labels
)

Map:   0%|          | 0/1050 [00:00<?, ? examples/s]

22 CHẠY INFERENCE VÀ EVALUATE
---------

In [80]:
metrics = trainer.evaluate(tokenized_val_dataset)

print(f"Loss      : {metrics['eval_loss']:.4f}")
print(f"Accuracy  : {metrics['eval_accuracy']:.4f}")
print(f"F1-score  : {metrics['eval_f1']:.4f}")
print(f"Precision : {metrics['eval_precision']:.4f}")
print(f"Recall    : {metrics['eval_recall']:.4f}")
print(f"Runtime   : {metrics['eval_runtime']:.2f} seconds")
print(f"Samples/s : {metrics['eval_samples_per_second']:.2f}")
print(f"Steps/s   : {metrics['eval_steps_per_second']:.2f}")
print(f"Epoch     : {metrics['epoch']}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Loss      : 0.2281
Accuracy  : 0.9611
F1-score  : 0.9608
Precision : 0.9606
Recall    : 0.9611
Runtime   : 4.64 seconds
Samples/s : 226.07
Steps/s   : 14.21
Epoch     : 5.0


22 DECODE LABELS
----------------

Chuyển labels từ giá trị 0 và 1 thành keep và normalize cho người đọc dễ nhìn.

In [81]:
id2label = {
    0: "KEEP",
    1: "NORMALIZE"
}

In [82]:
decoded_preds = []

for sent_preds in pred_ids:

    labels = [
        id2label[p.item()]
        for p in sent_preds
    ]

    decoded_preds.append(labels)

22 LƯU PREDICT VÀO DATAFRAME
----------------------------

Đưa kết quả đã predict vào lại dataframe validation xem model đã xử lý như thế nào.

In [86]:
df_val["pred_labels"] = decoded_preds

In [93]:
df_val[[
    "raw_tokens",
    "pred_labels"
]].head(10)

,raw_tokens,pred_labels
0,"[Bên, ni, đèo, thì, ""bả, sá"",, bên, tê, đèo, t...","[KEEP, KEEP, NORMALIZE, KEEP, KEEP, KEEP, KEEP..."
1,"[2, mày, bán, tao, ngồi, đếm, xiền, xem, có, đ...","[NORMALIZE, KEEP, KEEP, KEEP, KEEP, KEEP, KEEP..."
2,"[Vứt, con, đấy, đi, hành, hung, ng, khác, tron...","[NORMALIZE, KEEP, KEEP, KEEP, KEEP, KEEP, KEEP..."
3,"[V, là, ai, ở, nhà, ôm, con, cũng, rảnh, háng,...","[NORMALIZE, NORMALIZE, KEEP, KEEP, KEEP, KEEP,..."
4,"[3, đứa, m, có, v, k]","[NORMALIZE, KEEP, KEEP, NORMALIZE, KEEP, NORMA..."
5,"[Ngon,, vào, hốc, thoải, mái, k, phải, kiêng, ...","[KEEP, KEEP, KEEP, KEEP, KEEP, KEEP, NORMALIZE..."
6,"[giờ, nghĩ, lại, thấy, lớp, 6, tai, mình, cũng...","[KEEP, KEEP, KEEP, KEEP, KEEP, KEEP, KEEP, KEE..."
7,"[Đọc, tiêu, đề, ba, chấm, wua]","[NORMALIZE, KEEP, KEEP, KEEP, KEEP, KEEP, NORM..."
8,"[thành, ra, đến, h, yêu, đương, vẫn, phải, giấ...","[NORMALIZE, KEEP, KEEP, KEEP, NORMALIZE, KEEP,..."
9,"[Dơ, cái, nách, lên, coi, có, thâm, không, sao...","[NORMALIZE, KEEP, KEEP, KEEP, KEEP, KEEP, KEEP..."


23 SAVE MODEL
-------------

In [89]:
save_path = "/kaggle/working/final_phobert_multilexnorm"

In [90]:
trainer.save_model(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [91]:
tokenizer.save_pretrained(save_path)

('/kaggle/working/final_phobert_multilexnorm/tokenizer_config.json',
 '/kaggle/working/final_phobert_multilexnorm/vocab.txt',
 '/kaggle/working/final_phobert_multilexnorm/bpe.codes',
 '/kaggle/working/final_phobert_multilexnorm/added_tokens.json')

24 TẢI MODEL ĐÃ TRAIN VỀ MÁY
----------------------------

In [92]:
import shutil

shutil.make_archive(
    "/kaggle/working/final_phobert_multilexnorm",
    "zip",
    "/kaggle/working/final_phobert_multilexnorm"
)

'/kaggle/working/final_phobert_multilexnorm.zip'

CHECK EXACT MATCH
-----------------

Hiện tại giá trị predict đang ở dạng subword prediction. nên nếu check exact match với word label thì sẽ sai hoàn toàn. Vì thế nên ta phải covert về đúng word-level

Token lại

In [99]:
tokenized_inputs = []

for tokens in df_val["raw_tokens"]:

    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokenized_inputs.append(encoding)

In [100]:
predictions = trainer.predict(tokenized_val_dataset)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [101]:
preds = np.argmax(
    predictions.predictions,
    axis=-1
)

Align về word-level

In [104]:
word_level_preds = []

for pred_seq, label_seq in zip(
    preds,
    tokenized_val_dataset["labels"]
):

    filtered_preds = []

    for p, l in zip(pred_seq, label_seq):

        # chỉ lấy token thật
        if l != -100:

            filtered_preds.append(int(p))

    word_level_preds.append(filtered_preds)

Lưu vào dataframe

In [105]:
df_val["pred_labels"] = word_level_preds

In [106]:
df_val.head()

,raw,norm,lang,split,raw_tokens,norm_tokens,labels,pred_labels
0,"['Bên' 'ni' 'đèo' 'thì' '""bả' 'sá"",' 'bên' 'tê...","['Bên' 'này' 'đèo' 'thì' '""bả' 'sá"",' 'bên' 'k...",vi,validation,"[Bên, ni, đèo, thì, ""bả, sá"",, bên, tê, đèo, t...","[Bên, này, đèo, thì, ""bả, sá"",, bên, kia, đèo,...","[0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,['2' 'mày' 'bán' 'tao' 'ngồi' 'đếm' 'xiền' 'xe...,['2' 'mày' 'bán' 'tao' 'ngồi' 'đếm' 'tiền' 'xe...,vi,validation,"[2, mày, bán, tao, ngồi, đếm, xiền, xem, có, đ...","[2, mày, bán, tao, ngồi, đếm, tiền, xem, có, đ...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,['Vứt' 'con' 'đấy' 'đi' 'hành' 'hung' 'ng' 'kh...,['Vứt' 'con' 'đấy' 'đi' 'hành' 'hung' 'người' ...,vi,validation,"[Vứt, con, đấy, đi, hành, hung, ng, khác, tron...","[Vứt, con, đấy, đi, hành, hung, người, khác, t...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]"
3,['V' 'là' 'ai' 'ở' 'nhà' 'ôm' 'con' 'cũng' 'rả...,['Vậy' 'là' 'ai' 'ở' 'nhà' 'ôm' 'con' 'cũng' '...,vi,validation,"[V, là, ai, ở, nhà, ôm, con, cũng, rảnh, háng,...","[Vậy, là, ai, ở, nhà, ôm, con, cũng, rảnh, hán...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4,['3' 'đứa' 'm' 'có' 'v' 'k'],['3' 'đứa' 'm' 'có' 'vậy' 'không'],vi,validation,"[3, đứa, m, có, v, k]","[3, đứa, m, có, vậy, không]","[0, 0, 0, 0, 1, 1]","[0, 0, 1, 0, 1, 1]"


In [107]:
exact_matches = 0

for true_seq, pred_seq in zip(
    df_val["labels"],
    df_val["pred_labels"]
):

    pred_seq = pred_seq[:len(true_seq)]

    if true_seq == pred_seq:
        exact_matches += 1

exact_match = exact_matches / len(df_val)

print(f"Exact Match: {exact_match:.4f}")

Exact Match: 0.7086
